In [2]:
!pip install groq

In [3]:
import os
import time
import logging
from typing import List, Dict, Any, Tuple
from groq import Groq, APIError, RateLimitError, APITimeoutError
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)
logger = logging.getLogger("RAG-Generator")
class RAGGenerationPipeline:
    def __init__(self, api_key: str = None, model_name: str = "llama-3.3-70b-versatile"):
        """
        Initializes the LLM client.
        Note: Always check live provider documentation for currently active free
        model IDs to prevent 404 Not Found errors.
        """
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        if not self.api_key:
            raise ValueError("API key must be provided or set in environment variables (e.g., GROQ_API_KEY).")
        self.model_name = model_name
        self.client = Groq(api_key=self.api_key)
    def construct_prompt(self, query: str, retrieved_chunks: List[Dict[str, Any]]) -> Tuple[str, str]:
        """
        Constructs system instructions and context-injected user prompt.
        """
        system_prompt = (
            "You are a rigorous, fact-based AI assistant. Your objective is to answer the user's "
            "query strictly using *only* the provided context chunks below.\n\n"
            "Guidelines:\n"
            "1. Do not use external knowledge, prior assumptions, or extrapolate beyond the text.\n"
            "2. If the context does not contain enough evidence to answer the query, reply with: "
            "'I am sorry, but I cannot find the answer in the provided documents.'\n"
            "3. Ground your statements by referencing the source identifier where applicable."
        )
        context_blocks = []
        for i, chunk in enumerate(retrieved_chunks):
            source_id = chunk.get("id", f"Chunk-{i+1}")
            text = chunk.get("text", "").strip()
            context_blocks.append(f"--- Source [{source_id}] ---\n{text}")
        context_str = "\n\n".join(context_blocks)
        user_prompt = (
            f"Retrieved Context:\n{context_str}\n\n"
            f"User Query: {query}\n\n"
            f"Answer:"
        )
        return system_prompt, user_prompt
    def run(self, query: str, retrieved_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        Executes the generation step with latency tracking, out-of-domain detection,
        and robust retry/error handling.
        """
        start_time = time.perf_counter()
        if not retrieved_chunks:
            total_latency = round(time.perf_counter() - start_time, 4)
            logger.warning("Retrieval returned zero chunks. Triggering out-of-domain fallback.")
            return {
                "answer": "I am sorry, but I cannot find relevant information in the provided documents to answer your query.",
                "latency_seconds": total_latency,
                "status": "out_of_domain_empty"
            }
        system_prompt, user_prompt = self.construct_prompt(query, retrieved_chunks)
        max_retries = 3
        backoff_base = 2.0
        response_text = ""
        status = "success"

        for attempt in range(max_retries):
            try:
                gen_start = time.perf_counter()
                completion = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=0.1,
                    max_tokens=512,
                    timeout=15.0
                )
                gen_duration = time.perf_counter() - gen_start
                logger.info(f"Generation complete. Inference time: {gen_duration:.4f}s")
                response_text = completion.choices[0].message.content.strip()
                break
            except RateLimitError as e:
                logger.warning(f"Rate limit hit on attempt {attempt + 1}/{max_retries}: {e}")
                if attempt == max_retries - 1:
                    status = "rate_limit_exceeded"
                    response_text = "Error: Rate limit exceeded. Please try again shortly."
                else:
                    time.sleep(backoff_base ** attempt)
            except APITimeoutError as e:
                logger.warning(f"API timeout on attempt {attempt + 1}/{max_retries}: {e}")
                if attempt == max_retries - 1:
                    status = "timeout"
                    response_text = "Error: The request timed out. Please try again."
                else:
                    time.sleep(backoff_base ** attempt)
            except APIError as e:
                status = "api_error"
                logger.error(f"API Error encountered: {e}")
                if "404" in str(e):
                    response_text = f"Error: Model '{self.model_name}' was not found or has been deprecated. Verify live provider IDs."
                else:
                    response_text = f"Error: Provider API failed with code {e.status_code if hasattr(e, 'status_code') else 'Unknown'}."
                break
            except Exception as e:
                status = "unknown_error"
                logger.exception(f"Unexpected exception during generation: {e}")
                response_text = "Error: An unexpected error occurred while processing your request."
                break
        total_latency = round(time.perf_counter() - start_time, 4)
        logger.info(f"End-to-end RAG Pipeline Latency: {total_latency}s | Status: {status}")
        return {
            "answer": response_text,
            "latency_seconds": total_latency,
            "status": status
        }